# Prototype lớp ghép từ luật (RuleComposer)

Code: `zeroshot/nesy/compose.py` (mô hình), `protocol.py` (KG và cách đánh giá), `features.py` (trích feature).

Mỗi luật trong KG là "công thức" ghép prototype của lớp: text CLIP của lớp, cộng embedding các primitive được require, trừ embedding các primitive bị forbid. Luật thứ tự được chấm bằng cách so khớp theo thời gian. Lớp unseen được ghép từ cùng các embedding primitive đã học trên lớp seen.

Các biến thể, dùng chung encoder video và chung tập validation:
- `text`: không dùng luật, $t_y = c_y$. Đây là baseline.
- `rules`: đầy đủ.
- `rules_noseq`: bỏ luật thứ tự.

Cột kết quả:
- **clean**: epoch và γ chọn trên val.
- **`_best_test`**: lấy max trên test, cách các paper báo cáo.
- **`zsl_no_rules` / `zsl_shuffled`**: kiểm tra can thiệp luật trên lớp unseen.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, gc, json, subprocess, importlib
import numpy as np, pandas as pd, torch, torch.nn.functional as F

DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'
SKEL = {'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
        'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz'}
OUT_DIR   = f'{DRIVE_DIR}/compose'
# Cache feature bộ phận × thời gian (fp16): ~3.4 GB / split NTU60, ~7 GB / split NTU120.
# Mặc định để trên ổ local của Colab (mất khi hết phiên, trích lại ~45 phút cho 4 split).
# Drive còn chỗ thì đổi sang f'{DRIVE_DIR}/cache_p2' để lần sau khỏi trích.
CACHE_DIR = '/content/cache_p2'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
NESY = f'{REPO}/zeroshot/nesy'
if NESY not in sys.path:
    sys.path.insert(0, NESY)
import protocol, features, compose
for m in (protocol, features, compose):
    importlib.reload(m)
TASKS = list(protocol.UNSEEN)

In [ ]:
prims, names, R, seqs = protocol.load_kg(f'{REPO}/zeroshot/kg_rules.json', f'{REPO}/zeroshot/primitive.json')
prim_ids = [p['id'] for p in prims]
print(f'{len(prims)} primitive | mỗi lớp trung bình {(R > 0).sum(1).mean():.1f} require, {(R < 0).sum(1).mean():.1f} forbid | {len(seqs)} lớp có luật thứ tự')

# Primitive mà lớp unseen dùng nhưng không lớp seen nào dùng: embedding của nó giữ nguyên giá trị khởi tạo từ CLIP text.
for task in TASKS:
    unseen = protocol.UNSEEN[task]
    seen = [c for c in range(protocol.n_classes(task)) if c not in unseen]
    cov = pd.DataFrame(protocol.coverage(R, seen, unseen, prim_ids))
    cov.insert(1, 'name', [names[c] for c in cov.cls])
    print(f'\n=== {task}: {int((cov.n_unused_in_seen > 0).sum())}/{len(cov)} lớp unseen có primitive chỉ dựa vào CLIP text'
          f' | có luật thứ tự: {[names[c] for c in unseen if c in seqs]}')
    display(cov[cov.n_unused_in_seen > 0])

In [ ]:
# Text CLIP ViT-L/14@336 (cùng text encoder với Neuron).
#   class_text (120, 768): tên lớp + 5 file mô tả LLM của Neuron (mỗi lớp 10 câu/file), trung bình các nguồn
#   prim_text  (71, 768) : trung bình của 'description' trong primitive.json và cụm từ tạo từ id (vd 'hand at face')
TEXT_CACHE = f'{OUT_DIR}/text.npz'
OLD_CLASS_CACHE = f'{DRIVE_DIR}/phase1/text_proto.npz'   # nếu đã tạo ở lần chạy trước thì dùng lại
SRC = {'spatial_cg': 'spatial/spatial_cg.json', 'spatial_mg': 'spatial/spatial_mg.json',
       'spatial_fg': 'spatial/spatial_fg.json', 'temporal_fp': 'temporal/temporal_fp.json',
       'temporal_sp': 'temporal/temporal_sp.json'}
unit = lambda a: a / np.linalg.norm(a, axis=-1, keepdims=True)

if os.path.exists(TEXT_CACHE):
    t = np.load(TEXT_CACHE)
    class_text, prim_text = t['class_text'], t['prim_text']
else:
    from transformers import CLIPModel, CLIPTokenizer
    cid = 'openai/clip-vit-large-patch14-336'
    clip = CLIPModel.from_pretrained(cid, torch_dtype=torch.float16).cuda().eval()
    tok = CLIPTokenizer.from_pretrained(cid)

    @torch.no_grad()
    def enc(texts, bs=256):
        out = []
        for i in range(0, len(texts), bs):
            b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=77, return_tensors='pt').to('cuda')
            f = clip.get_text_features(**b)
            if not torch.is_tensor(f):   # một số bản transformers trả về object thay vì tensor
                f = f.text_embeds if getattr(f, 'text_embeds', None) is not None else f.pooler_output
            out.append(F.normalize(f.float(), dim=-1).cpu())
        return torch.cat(out).numpy()

    if os.path.exists(OLD_CLASS_CACHE):
        class_text = unit(np.load(OLD_CLASS_CACHE)['per_src'].mean(1))
    else:
        per_src = [enc([f'a person {n}' for n in names])]
        for s, rel in SRC.items():
            d = json.load(open(f'{NESY}/semantics/ntu/{rel}', encoding='utf-8'))
            assert len(d) == 120, (s, len(d))     # thứ tự key = thứ tự lớp 1..120
            per_src.append(unit(np.stack([enc(d[k]).mean(0) for k in d])))
        class_text = unit(np.stack(per_src, 1).mean(1))
    phrase = [p['id'].replace('.', ' ').replace('_', ' ') for p in prims]
    prim_text = unit(enc([p['description'] for p in prims]) + enc(phrase))
    np.savez(TEXT_CACHE, class_text=class_text, prim_text=prim_text)
    del clip
print('class_text', class_text.shape, '| prim_text', prim_text.shape)

In [ ]:
# Trích cube 8 bộ phận × 16 thời gian × 256 bằng encoder Neuron đóng băng (weights riêng cho từng split).
# Có cache rồi thì bỏ qua.
importlib.reload(features)
for ds in ('ntu60', 'ntu120'):
    tasks = [t for t in TASKS if t.startswith(ds + '_')]
    if all(os.path.exists(features.cache_paths(CACHE_DIR, t, 'test')['label']) for t in tasks):
        print(ds, ': cache có sẵn'); continue
    skel = dict(np.load(SKEL[ds]))       # đọc một lần cho cả dataset (7-20 GB RAM)
    for t in tasks:
        features.extract_split(skel, WEIGHTS_DIR, t, protocol.UNSEEN[t], CACHE_DIR)
    del skel; gc.collect()
!du -sh {CACHE_DIR}

In [ ]:
for m in (protocol, compose):
    importlib.reload(m)
SEEDS = [0]
VARIANTS = {'text': dict(use_rules=False), 'rules': dict(), 'rules_noseq': dict(use_seq=False)}
MODEL_KW = dict(d=256, n_layers=2, n_heads=4, dropout=0.1, p_drop=0.2, lam_reg=1.0)
TRAIN_KW = dict(epochs=30, lr=5e-4, wd=1e-4, bs=256)
RES_CSV = f'{OUT_DIR}/results.csv'

results, hists = [], {}
for task in TASKS:
    data = features.load_cache(CACHE_DIR, task, kind='cube')
    for seed in SEEDS:
        for name, extra in VARIANTS.items():
            make = compose.make_composer(class_text, prim_text, R, seqs, **MODEL_KW, **extra)
            res, hist = protocol.run_split(data, task, make, name=name, seed=seed, train_kw=TRAIN_KW,
                                           extra_eval=compose.rule_interventions(R, seqs))
            results.append(res)
            hists[(task, name, seed)] = hist
            pd.DataFrame(results).to_csv(RES_CSV, index=False)
    del data; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(results)
show = ['task', 'model', 'seed', 'epoch_val', 'val_zsl', 'zsl', 'zsl_best_test', 'gzsl_H', 'H_best_test',
        'zsl_no_rules', 'zsl_shuffled']
display(df[[c for c in show if c in df]].round(4))

In [ ]:
# Phần đóng góp của KG: rules - text, ở cả hai cách tính
rows = []
for (task, seed), g in df.groupby(['task', 'seed'], sort=False):
    r = {m: g[g.model == m].iloc[0] for m in g.model}
    if 'text' not in r or 'rules' not in r:
        continue
    row = dict(task=task, seed=seed,
               zsl_text=r['text'].zsl, zsl_rules=r['rules'].zsl, gain=r['rules'].zsl - r['text'].zsl,
               zsl_text_bt=r['text'].zsl_best_test, zsl_rules_bt=r['rules'].zsl_best_test,
               gain_bt=r['rules'].zsl_best_test - r['text'].zsl_best_test,
               H_text=r['text'].gzsl_H, H_rules=r['rules'].gzsl_H,
               H_text_bt=r['text'].H_best_test, H_rules_bt=r['rules'].H_best_test,
               no_rules=r['rules'].zsl_no_rules, shuffled=r['rules'].zsl_shuffled)
    if 'rules_noseq' in r:
        row['zsl_noseq'] = r['rules_noseq'].zsl
    rows.append(row)
summary = pd.DataFrame(rows).round(4)
summary.to_csv(f'{OUT_DIR}/summary.csv', index=False)
display(summary)

In [ ]:
# Nét liền: val ZSL (chọn epoch). Nét đứt: test ZSL của lần train cuối.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(TASKS), figsize=(4.5 * len(TASKS), 3.2))
for ax, task in zip(axes, TASKS):
    for i, n in enumerate(VARIANTS):
        h = hists.get((task, n, SEEDS[0]))
        if h:
            ax.plot([r['epoch'] for r in h['val']], [r['val_zsl'] for r in h['val']], color=f'C{i}', label=n)
            ax.plot([r['epoch'] for r in h['test']], [r['test_zsl'] for r in h['test']], color=f'C{i}', ls='--')
    ax.set_title(task.replace('_xsub_', ' '), fontsize=9)
    ax.set_xlabel('epoch')
axes[0].set_ylabel('ZSL')
axes[0].legend()
plt.tight_layout()
plt.show()

## Đọc kết quả

- **`gain` / `gain_bt`**: hiệu `rules` − `text`, tức phần đóng góp của KG. Đây là con số chính.
- **`no_rules` và `shuffled` so với `zsl_rules`**: dùng cùng mô hình `rules`, chỉ xóa hoặc xáo trộn luật của các lớp unseen. Nếu cả hai đều giảm rõ thì mô hình thật sự dùng luật để nhận ra lớp unseen.
- **`zsl_noseq` so với `zsl_rules`**: phần đóng góp của luật thứ tự.

Gửi mình bảng ở cell `summary` và hình ở cell `curves`.